# 25 — Research-Grade Experiments, Ablations, and Patient-Grouped Cross-Validation

In the previous notebook, we built a realistic ultrasound classification pipeline using:

- Metadata CSV files
- Patient-level splits
- File-path based datasets
- Training-only normalization
- Class-imbalance strategies
- Site/device-aware evaluation
- Harmonization baselines
- Transfer learning
- Patient-level prediction aggregation

Now we will study how to turn that pipeline into a **research-grade experimental protocol**.

A single train/validation split can be useful for development, but it may also be unstable—especially when the dataset is small.

Research questions often require stronger evidence.

## In this notebook, we will study:

1. Why one train/validation split may be unstable
2. Patient-grouped cross-validation
3. Stratified group folds
4. Repeated experiments with multiple seeds
5. Hyperparameter tuning without leakage
6. Nested-validation intuition
7. Ablation studies
8. Comparing preprocessing strategies fairly
9. Comparing harmonization methods fairly
10. Confidence intervals
11. Patient-level bootstrap
12. External validation
13. Statistical model comparison intuition
14. Experiment tracking
15. Building reproducible result tables
16. Reporting research-quality deep-learning experiments

## Main Goal

A stronger research workflow looks like:

$$
\boxed{
\text{Patient Groups}
\rightarrow
\text{Cross-Validation}
\rightarrow
\text{Controlled Experiments}
\rightarrow
\text{Model Selection}
\rightarrow
\text{External Validation}
}
$$

The central principle is:

> **The unit used for splitting, resampling, uncertainty estimation, and evaluation should match the independent scientific unit whenever possible.**

For many ultrasound studies, that unit is the:

$$
\boxed{
Patient
}
$$


In [ ]:
import copy
import hashlib
import json
import math
import random
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import (
    Dataset,
    DataLoader
)

print("PyTorch:", torch.__version__)


# 1. Why One Train/Validation Split May Be Unstable

Suppose you have:

$$
60
$$

patients.

A single split may place:

- Easier patients in validation
- Harder patients in training
- One scanner mostly in validation
- One rare class unevenly across sets

Then your measured validation score may depend strongly on the particular split.

This is especially important when:

- Dataset size is small
- Classes are imbalanced
- There are multiple sites/devices
- Patient appearance is heterogeneous


# 2. The Goal of Cross-Validation

Cross-validation asks:

> Does the model perform consistently when different patient groups are held out?

In:

$$
K
$$

-fold cross-validation, every patient belongs to one fold.

For each run:

- Train on $K-1$ folds
- Validate on the remaining fold

Repeat until every fold has served as validation.


# 3. Patient-Grouped Cross-Validation

The key rule is:

$$
\boxed{
All\ images\ from\ one\ patient
\rightarrow
Same\ fold
}
$$

If patient P001 has:

$$
10
$$

ultrasound images, all 10 must remain together.

Otherwise correlated images can leak across training and validation.


# 4. Why Ordinary Image-Level K-Fold Can Be Wrong

Image-level K-fold may produce:

$$
P001\ image_1
\rightarrow
Train
$$

and:

$$
P001\ image_2
\rightarrow
Validation
$$

The network may recognize patient-specific or acquisition-specific characteristics.

The resulting validation score can become overly optimistic.


# 5. Synthetic Patient-Grouped Ultrasound Dataset

To keep this notebook self-contained, we will generate a small synthetic ultrasound-like dataset in memory.

We will create:

- Internal development patients
- Multiple images per patient
- Three classes
- Multiple internal sites/devices
- A completely separate external site

The goal is to study **experimental methodology**, not image realism.


In [ ]:
def make_research_image(
    class_index,
    image_size,
    site,
    device,
    generator
):
    image = torch.zeros(
        1,
        image_size,
        image_size
    )

    center = image_size // 2

    shift_y = int(
        torch.randint(
            -4,
            5,
            (1,),
            generator=generator
        ).item()
    )

    shift_x = int(
        torch.randint(
            -4,
            5,
            (1,),
            generator=generator
        ).item()
    )

    cy = center + shift_y
    cx = center + shift_x

    if class_index == 0:
        image[
            :,
            6:image_size - 6,
            max(0, cx - 2):
            min(image_size, cx + 3)
        ] = 0.9

    elif class_index == 1:
        image[
            :,
            max(0, cy - 2):
            min(image_size, cy + 3),
            6:image_size - 6
        ] = 0.9

    elif class_index == 2:
        image[
            :,
            max(0, cy - 7):
            min(image_size, cy + 8),
            max(0, cx - 7):
            min(image_size, cx + 8)
        ] = 0.9

    else:
        raise ValueError(
            "class_index must be 0, 1, or 2"
        )

    noise = torch.randn(
        image.shape,
        generator=generator
    ) * 0.10

    image = (
        image
        + noise
    ).clamp(
        0.0,
        1.0
    )

    site_gain = {
        "Site_A": 0.90,
        "Site_B": 1.00,
        "Site_C": 1.08,
        "External_Site": 0.82,
    }[
        site
    ]

    device_offset = {
        "Device_A": 0.00,
        "Device_B": 0.04,
        "Device_C": 0.08,
        "Device_X": 0.12,
    }[
        device
    ]

    image = (
        image
        * site_gain
        + device_offset
    ).clamp(
        0.0,
        1.0
    )

    return image


# 6. Build Internal and External Patient Cohorts

The external site will not participate in:

- Fold construction
- Hyperparameter tuning
- Ablation selection
- Threshold/model selection

It is reserved for later external validation.


In [ ]:
def build_research_dataset(
    num_internal_patients=75,
    num_external_patients=18,
    images_per_patient=2,
    image_size=32
):
    images = []
    records = []

    internal_sites = [
        "Site_A",
        "Site_B",
        "Site_C"
    ]

    internal_devices = [
        "Device_A",
        "Device_B",
        "Device_C"
    ]

    for patient_index in range(
        num_internal_patients
    ):
        patient_id = (
            f"I{patient_index:03d}"
        )

        label = (
            patient_index
            % 3
        )

        site = internal_sites[
            (patient_index // 3)
            % len(
                internal_sites
            )
        ]

        device = internal_devices[
            (
                patient_index
                + patient_index // 3
            )
            % len(
                internal_devices
            )
        ]

        generator = (
            torch.Generator()
            .manual_seed(
                1000
                + patient_index
            )
        )

        for image_index in range(
            images_per_patient
        ):
            image = make_research_image(
                label,
                image_size,
                site,
                device,
                generator
            )

            tensor_index = len(
                images
            )

            images.append(
                image
            )

            records.append({
                "tensor_index":
                    tensor_index,

                "patient_id":
                    patient_id,

                "label":
                    label,

                "site":
                    site,

                "device":
                    device,

                "cohort":
                    "internal",

                "image_index":
                    image_index
            })

    for patient_index in range(
        num_external_patients
    ):
        patient_id = (
            f"E{patient_index:03d}"
        )

        label = (
            patient_index
            % 3
        )

        site = (
            "External_Site"
        )

        device = (
            "Device_X"
        )

        generator = (
            torch.Generator()
            .manual_seed(
                5000
                + patient_index
            )
        )

        for image_index in range(
            images_per_patient
        ):
            image = make_research_image(
                label,
                image_size,
                site,
                device,
                generator
            )

            tensor_index = len(
                images
            )

            images.append(
                image
            )

            records.append({
                "tensor_index":
                    tensor_index,

                "patient_id":
                    patient_id,

                "label":
                    label,

                "site":
                    site,

                "device":
                    device,

                "cohort":
                    "external",

                "image_index":
                    image_index
            })

    return (
        images,
        pd.DataFrame(
            records
        )
    )


research_images, research_metadata = (
    build_research_dataset()
)

print(
    "Images:",
    len(
        research_images
    )
)

print(
    "Rows:",
    len(
        research_metadata
    )
)


# 7. Inspect Cohorts


In [ ]:
print(
    research_metadata[
        "cohort"
    ].value_counts()
)

print()

print(
    pd.crosstab(
        research_metadata[
            "cohort"
        ],
        research_metadata[
            "label"
        ]
    )
)


# 8. Build One Row Per Patient

Because each patient has one class label in this example, fold assignment can happen at the patient level.


In [ ]:
internal_metadata = (
    research_metadata[
        research_metadata[
            "cohort"
        ]
        == "internal"
    ]
    .reset_index(
        drop=True
    )
)

external_metadata = (
    research_metadata[
        research_metadata[
            "cohort"
        ]
        == "external"
    ]
    .reset_index(
        drop=True
    )
)

patient_table = (
    internal_metadata
    .groupby(
        "patient_id",
        as_index=False
    )
    .agg({
        "label":
            "first",

        "site":
            "first",

        "device":
            "first"
    })
)

print(
    patient_table.head()
)

print(
    "Internal patients:",
    len(
        patient_table
    )
)


# 9. Stratified Group Folds

We want two properties:

## Grouping

All images from one patient stay together.

## Stratification

Each fold has a similar class distribution.

Because each patient has one class label here, we can:

1. Shuffle patients within each class
2. Assign them round-robin across folds


In [ ]:
def make_stratified_group_folds(
    patient_table,
    n_splits=5,
    seed=42
):
    patient_to_fold = {}

    labels = sorted(
        patient_table[
            "label"
        ].unique()
    )

    for class_index in labels:
        class_patients = (
            patient_table[
                patient_table[
                    "label"
                ]
                == class_index
            ][
                "patient_id"
            ]
            .tolist()
        )

        rng = random.Random(
            seed
            + int(
                class_index
            )
        )

        rng.shuffle(
            class_patients
        )

        for position, patient_id in enumerate(
            class_patients
        ):
            patient_to_fold[
                patient_id
            ] = (
                position
                % n_splits
            )

    return patient_to_fold


patient_to_fold = (
    make_stratified_group_folds(
        patient_table,
        n_splits=5,
        seed=42
    )
)

patient_table[
    "fold"
] = patient_table[
    "patient_id"
].map(
    patient_to_fold
)

print(
    patient_table.head()
)


# 10. Verify Every Patient Has Exactly One Fold


In [ ]:
assert (
    patient_table[
        "patient_id"
    ].nunique()
    ==
    len(
        patient_to_fold
    )
)

assert not patient_table[
    "fold"
].isna().any()

print(
    "Every patient has one fold."
)


# 11. Inspect Class Balance Across Folds


In [ ]:
fold_class_table = pd.crosstab(
    patient_table[
        "fold"
    ],
    patient_table[
        "label"
    ]
)

print(
    fold_class_table
)


# 12. Attach Fold IDs to Every Image


In [ ]:
internal_metadata[
    "fold"
] = internal_metadata[
    "patient_id"
].map(
    patient_to_fold
)

print(
    internal_metadata[
        [
            "patient_id",
            "label",
            "fold"
        ]
    ].head()
)


# 13. Verify No Patient Appears in Multiple Folds


In [ ]:
folds_per_patient = (
    internal_metadata
    .groupby(
        "patient_id"
    )[
        "fold"
    ]
    .nunique()
)

assert (
    folds_per_patient
    == 1
).all()

print(
    "Patient grouping verified."
)


# 14. A Note on `StratifiedGroupKFold`

For more complex real datasets, scikit-learn provides:

```python
StratifiedGroupKFold
```

This is useful when you want:

- Stratification
- Group independence
- Standardized split utilities

The custom round-robin method here is intentionally transparent for teaching.

For real research, inspect every generated fold rather than trusting the splitter blindly.


# 15. Cross-Validation Visualization

For:

$$
K=5
$$

the protocol is:

$$
\begin{array}{|c|c|}
\hline
Run\ 1 & Fold\ 0\ validation,\ others\ training \\
\hline
Run\ 2 & Fold\ 1\ validation,\ others\ training \\
\hline
Run\ 3 & Fold\ 2\ validation,\ others\ training \\
\hline
Run\ 4 & Fold\ 3\ validation,\ others\ training \\
\hline
Run\ 5 & Fold\ 4\ validation,\ others\ training \\
\hline
\end{array}
$$


# 16. Training-Only Preprocessing Inside Every Fold

This is critical.

For fold 0:

$$
\mu_0,\sigma_0
$$

must be computed from the fold-0 **training patients only**.

For fold 1:

$$
\mu_1,\sigma_1
$$

must be recomputed from fold-1 training patients.

Do not compute one global mean/std from all internal patients before CV.

That leaks validation-fold information.


In [ ]:
def compute_tensor_mean_std(
    metadata,
    images
):
    tensors = torch.stack([
        images[
            int(
                tensor_index
            )
        ]
        for tensor_index in metadata[
            "tensor_index"
        ].tolist()
    ])

    mean = tensors.mean().item()
    std = tensors.std().item()

    return (
        mean,
        std
    )


# 17. Fold-Specific Preprocessing Example


In [ ]:
example_val_fold = 0

fold_train_metadata = (
    internal_metadata[
        internal_metadata[
            "fold"
        ]
        != example_val_fold
    ]
)

fold_val_metadata = (
    internal_metadata[
        internal_metadata[
            "fold"
        ]
        == example_val_fold
    ]
)

fold_mean, fold_std = (
    compute_tensor_mean_std(
        fold_train_metadata,
        research_images
    )
)

print(
    "Fold-specific mean:",
    fold_mean
)

print(
    "Fold-specific std:",
    fold_std
)


# 18. Research Dataset Class

This dataset supports different preprocessing/ablation modes.

Available normalization strategies:

- `global` — fold-training mean/std
- `per_image` — standardize each image independently
- `none` — no standardization

Training augmentation can also be turned on/off.


In [ ]:
class ResearchUltrasoundDataset(
    Dataset
):
    def __init__(
        self,
        metadata,
        images,
        training,
        normalization,
        train_mean=None,
        train_std=None
    ):
        self.metadata = (
            metadata
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.images = (
            images
        )

        self.training = (
            training
        )

        self.normalization = (
            normalization
        )

        self.train_mean = (
            train_mean
        )

        self.train_std = (
            train_std
        )

    def __len__(self):
        return len(
            self.metadata
        )

    def __getitem__(
        self,
        index
    ):
        row = self.metadata.iloc[
            index
        ]

        image = self.images[
            int(
                row[
                    "tensor_index"
                ]
            )
        ].clone()

        if self.training:
            shift = int(
                torch.randint(
                    -2,
                    3,
                    (1,)
                ).item()
            )

            image = torch.roll(
                image,
                shifts=shift,
                dims=2
            )

        if self.normalization == "global":
            image = (
                image
                - self.train_mean
            ) / (
                self.train_std
                + 1e-8
            )

        elif self.normalization == "per_image":
            image = (
                image
                - image.mean()
            ) / (
                image.std()
                + 1e-8
            )

        elif self.normalization == "none":
            pass

        else:
            raise ValueError(
                "Unknown normalization mode."
            )

        target = torch.tensor(
            int(
                row[
                    "label"
                ]
            ),
            dtype=torch.long
        )

        return (
            image,
            target,
            index
        )


# 19. Small Research CNN

We keep the model intentionally compact because the focus is experimental design.


In [ ]:
class ResearchCNN(nn.Module):
    def __init__(
        self,
        num_classes=3,
        width=12,
        dropout=0.0
    ):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                1,
                width,
                3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                width,
                width * 2,
                3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                width * 2,
                width * 4,
                3,
                padding=1
            ),
            nn.ReLU()
        )

        self.pool = (
            nn.AdaptiveAvgPool2d(
                1
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Linear(
            width * 4,
            num_classes
        )

    def forward(
        self,
        x
    ):
        x = self.features(
            x
        )

        x = self.pool(
            x
        )

        x = torch.flatten(
            x,
            start_dim=1
        )

        x = self.dropout(
            x
        )

        return self.classifier(
            x
        )


# 20. Metric Helpers


In [ ]:
def confusion_matrix_multiclass(
    targets,
    predictions,
    num_classes
):
    matrix = torch.zeros(
        num_classes,
        num_classes,
        dtype=torch.long
    )

    for target, prediction in zip(
        targets,
        predictions
    ):
        matrix[
            int(
                target
            ),
            int(
                prediction
            )
        ] += 1

    return matrix


def macro_f1_from_predictions(
    targets,
    predictions,
    num_classes
):
    confusion = (
        confusion_matrix_multiclass(
            targets,
            predictions,
            num_classes
        )
    )

    f1_values = []

    for class_index in range(
        num_classes
    ):
        tp = confusion[
            class_index,
            class_index
        ].item()

        fp = (
            confusion[
                :,
                class_index
            ].sum().item()
            - tp
        )

        fn = (
            confusion[
                class_index,
                :
            ].sum().item()
            - tp
        )

        precision = (
            tp
            / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )

        recall = (
            tp
            / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )

        f1 = (
            2
            * precision
            * recall
            / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

        f1_values.append(
            f1
        )

    return sum(
        f1_values
    ) / len(
        f1_values
    )


# 21. Device Setup


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device
)


# 22. Reproducibility Helper


In [ ]:
def set_seed(
    seed
):
    random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )


# 23. Training One Epoch


In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for images, targets, _ in loader:
        images = images.to(
            device
        )

        targets = targets.to(
            device
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            images
        )

        loss = criterion(
            logits,
            targets
        )

        loss.backward()

        optimizer.step()

        batch_size = (
            targets.size(0)
        )

        total_loss += (
            loss.item()
            * batch_size
        )

        total_samples += (
            batch_size
        )

    return (
        total_loss
        / total_samples
    )


# 24. Validation With Predictions


In [ ]:
def evaluate_with_predictions(
    model,
    dataset,
    loader,
    criterion,
    device,
    num_classes=3
):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    rows = []

    with torch.inference_mode():
        for images, targets, indices in loader:
            images = images.to(
                device
            )

            targets_device = targets.to(
                device
            )

            logits = model(
                images
            )

            loss = criterion(
                logits,
                targets_device
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            ).cpu()

            predictions = logits.argmax(
                dim=1
            ).cpu()

            batch_size = (
                targets.size(0)
            )

            total_loss += (
                loss.item()
                * batch_size
            )

            total_samples += (
                batch_size
            )

            for position in range(
                batch_size
            ):
                local_index = int(
                    indices[
                        position
                    ].item()
                )

                metadata_row = (
                    dataset
                    .metadata
                    .iloc[
                        local_index
                    ]
                )

                record = {
                    "patient_id":
                        metadata_row[
                            "patient_id"
                        ],

                    "true_label":
                        int(
                            targets[
                                position
                            ].item()
                        ),

                    "predicted_label":
                        int(
                            predictions[
                                position
                            ].item()
                        )
                }

                for class_index in range(
                    num_classes
                ):
                    record[
                        f"prob_class_{class_index}"
                    ] = float(
                        probabilities[
                            position,
                            class_index
                        ].item()
                    )

                rows.append(
                    record
                )

    prediction_df = pd.DataFrame(
        rows
    )

    accuracy = (
        prediction_df[
            "true_label"
        ].to_numpy()
        ==
        prediction_df[
            "predicted_label"
        ].to_numpy()
    ).mean()

    macro_f1 = (
        macro_f1_from_predictions(
            prediction_df[
                "true_label"
            ].tolist(),
            prediction_df[
                "predicted_label"
            ].tolist(),
            num_classes
        )
    )

    return (
        total_loss
        / total_samples,
        float(
            accuracy
        ),
        float(
            macro_f1
        ),
        prediction_df
    )


# 25. Build Fold DataLoaders

Each fold must fit preprocessing from that fold's training subset only.


In [ ]:
def build_fold_dataloaders(
    internal_metadata,
    images,
    val_fold,
    normalization,
    augmentation,
    batch_size=24
):
    train_metadata = (
        internal_metadata[
            internal_metadata[
                "fold"
            ]
            != val_fold
        ]
        .reset_index(
            drop=True
        )
    )

    val_metadata = (
        internal_metadata[
            internal_metadata[
                "fold"
            ]
            == val_fold
        ]
        .reset_index(
            drop=True
        )
    )

    train_mean, train_std = (
        compute_tensor_mean_std(
            train_metadata,
            images
        )
    )

    train_dataset = (
        ResearchUltrasoundDataset(
            train_metadata,
            images,
            training=augmentation,
            normalization=normalization,
            train_mean=train_mean,
            train_std=train_std
        )
    )

    val_dataset = (
        ResearchUltrasoundDataset(
            val_metadata,
            images,
            training=False,
            normalization=normalization,
            train_mean=train_mean,
            train_std=train_std
        )
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=0
    )

    return (
        train_dataset,
        val_dataset,
        train_loader,
        val_loader,
        train_mean,
        train_std
    )


# 26. Experiment Configuration

A research experiment should be defined by an explicit configuration.

Example:


In [ ]:
baseline_config = {
    "name":
        "baseline",

    "normalization":
        "global",

    "augmentation":
        True,

    "learning_rate":
        1e-3,

    "weight_decay":
        1e-4,

    "width":
        12,

    "dropout":
        0.0,

    "class_weighting":
        False
}

print(
    json.dumps(
        baseline_config,
        indent=2
    )
)


# 27. Compute Fold-Specific Class Weights

If class weighting is enabled, calculate weights from that fold's **training patients/images only**.

Never use the validation fold to calculate training weights.


In [ ]:
def class_weights_from_metadata(
    metadata,
    num_classes=3
):
    counts = torch.tensor([
        int(
            (
                metadata[
                    "label"
                ]
                == class_index
            ).sum()
        )
        for class_index in range(
            num_classes
        )
    ],
    dtype=torch.float32)

    weights = (
        counts.sum()
        / (
            num_classes
            * counts.clamp_min(
                1.0
            )
        )
    )

    return weights


# 28. Run One Fold Experiment

This function performs:

1. Fold-specific preprocessing
2. Model initialization
3. Training
4. Best-validation-loss checkpointing
5. Validation prediction collection


In [ ]:
def run_single_fold(
    internal_metadata,
    images,
    val_fold,
    seed,
    config,
    epochs=3,
    num_classes=3
):
    set_seed(
        seed
    )

    (
        train_dataset,
        val_dataset,
        train_loader,
        val_loader,
        train_mean,
        train_std
    ) = build_fold_dataloaders(
        internal_metadata,
        images,
        val_fold,
        normalization=config[
            "normalization"
        ],
        augmentation=config[
            "augmentation"
        ]
    )

    model = ResearchCNN(
        num_classes=num_classes,
        width=config[
            "width"
        ],
        dropout=config[
            "dropout"
        ]
    ).to(
        device
    )

    if config[
        "class_weighting"
    ]:
        weights = class_weights_from_metadata(
            train_dataset.metadata,
            num_classes=num_classes
        ).to(
            device
        )

        criterion = nn.CrossEntropyLoss(
            weight=weights
        )

    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config[
            "learning_rate"
        ],
        weight_decay=config[
            "weight_decay"
        ]
    )

    best_val_loss = float(
        "inf"
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )

    best_epoch = 0

    for epoch in range(
        epochs
    ):
        train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        (
            val_loss,
            val_accuracy,
            val_macro_f1,
            prediction_df
        ) = evaluate_with_predictions(
            model,
            val_dataset,
            val_loader,
            criterion,
            device,
            num_classes=num_classes
        )

        if val_loss < best_val_loss:
            best_val_loss = (
                val_loss
            )

            best_epoch = (
                epoch + 1
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

    model.load_state_dict(
        best_state
    )

    (
        val_loss,
        val_accuracy,
        val_macro_f1,
        prediction_df
    ) = evaluate_with_predictions(
        model,
        val_dataset,
        val_loader,
        criterion,
        device,
        num_classes=num_classes
    )

    prediction_df[
        "fold"
    ] = val_fold

    prediction_df[
        "seed"
    ] = seed

    prediction_df[
        "experiment"
    ] = config[
        "name"
    ]

    result = {
        "experiment":
            config[
                "name"
            ],

        "fold":
            val_fold,

        "seed":
            seed,

        "best_epoch":
            best_epoch,

        "val_loss":
            val_loss,

        "val_accuracy":
            val_accuracy,

        "val_macro_f1":
            val_macro_f1,

        "train_mean":
            train_mean,

        "train_std":
            train_std
    }

    return (
        result,
        prediction_df
    )


# 29. Quick One-Fold Demonstration


In [ ]:
demo_result, demo_predictions = (
    run_single_fold(
        internal_metadata,
        research_images,
        val_fold=0,
        seed=11,
        config=baseline_config,
        epochs=2
    )
)

print(
    demo_result
)


# 30. Repeated Cross-Validation

One cross-validation pass still uses one random initialization per fold.

Deep networks can vary because of:

- Weight initialization
- Batch order
- Stochastic augmentation
- Optimizer dynamics

So important experiments may be repeated using several random seeds.


# 31. Repeated Experiment Design

Example:

$$
5\ folds
\times
3\ seeds
=
15\ training\ runs
$$

For a serious project, you might use:

- More seeds
- More folds
- Repeated grouped CV

depending on computational budget.


In [ ]:
def run_cross_validation(
    internal_metadata,
    images,
    config,
    folds,
    seeds,
    epochs=3
):
    results = []
    predictions = []

    for seed in seeds:
        for fold in folds:
            result, prediction_df = (
                run_single_fold(
                    internal_metadata,
                    images,
                    val_fold=fold,
                    seed=seed,
                    config=config,
                    epochs=epochs
                )
            )

            results.append(
                result
            )

            predictions.append(
                prediction_df
            )

    return (
        pd.DataFrame(
            results
        ),
        pd.concat(
            predictions,
            ignore_index=True
        )
    )


# 32. Quick Repeated-CV Demo

For teaching, we use:

- 3 folds
- 2 seeds
- 2 epochs

For research, change to all folds and more seeds.


In [ ]:
quick_cv_results, quick_cv_predictions = (
    run_cross_validation(
        internal_metadata,
        research_images,
        baseline_config,
        folds=[
            0,
            1,
            2
        ],
        seeds=[
            11,
            22
        ],
        epochs=2
    )
)

print(
    quick_cv_results
)


# 33. Summarize Repeated Results

A useful summary reports:

- Mean
- Standard deviation
- Number of runs

Do not report only the single best fold.


In [ ]:
summary_table = (
    quick_cv_results[
        [
            "val_accuracy",
            "val_macro_f1"
        ]
    ]
    .agg([
        "mean",
        "std"
    ])
)

print(
    summary_table
)


# 34. Fold Variation vs Seed Variation

Two different uncertainties are present:

## Fold variation

Different patients are held out.

## Seed variation

Same experiment changes due to stochastic training.

Both can matter.

A research report should state clearly what was repeated.


# 35. Hyperparameter Tuning Without Leakage

Suppose fold 0 is the outer validation fold.

Wrong:

> Try many hyperparameters and select the best using fold 0 repeatedly.

Then fold 0 is no longer a clean evaluation fold.

Instead, tuning should happen **inside the outer training patients**.


# 36. Nested-Validation Intuition

Nested validation uses:

## Outer loop

Estimates generalization.

## Inner loop

Selects hyperparameters using only outer-training patients.

Conceptually:

$$
\boxed{
Outer\ Train
\rightarrow
Inner\ Tuning
\rightarrow
Choose\ Config
\rightarrow
Evaluate\ on\ Outer\ Validation
}
$$


# 37. Outer vs Inner Data

For one outer fold:

$$
Outer\ Validation
=
Fold\ k
$$

All remaining patients form:

$$
Outer\ Training
$$

Then the outer-training patients are split again into inner folds.

The outer validation patients remain completely untouched during tuning.


In [ ]:
def make_nested_patient_tables(
    patient_table,
    outer_fold,
    n_inner_splits=3,
    seed=123
):
    outer_train = (
        patient_table[
            patient_table[
                "fold"
            ]
            != outer_fold
        ]
        .reset_index(
            drop=True
        )
    )

    outer_val = (
        patient_table[
            patient_table[
                "fold"
            ]
            == outer_fold
        ]
        .reset_index(
            drop=True
        )
    )

    inner_map = (
        make_stratified_group_folds(
            outer_train[
                [
                    "patient_id",
                    "label",
                    "site",
                    "device"
                ]
            ],
            n_splits=n_inner_splits,
            seed=seed
        )
    )

    outer_train = outer_train.copy()

    outer_train[
        "inner_fold"
    ] = outer_train[
        "patient_id"
    ].map(
        inner_map
    )

    return (
        outer_train,
        outer_val
    )


outer_train_patients, outer_val_patients = (
    make_nested_patient_tables(
        patient_table,
        outer_fold=0
    )
)

print(
    pd.crosstab(
        outer_train_patients[
            "inner_fold"
        ],
        outer_train_patients[
            "label"
        ]
    )
)


# 38. Nested CV Can Be Expensive

If you have:

$$
5
$$

outer folds,

$$
3
$$

inner folds,

and:

$$
10
$$

hyperparameter configurations,

you already need:

$$
5\times3\times10
=
150
$$

training runs before considering multiple seeds.

Therefore research design must balance:

- Statistical rigor
- Compute budget
- Environmental cost
- Practical value


# 39. Hyperparameter Search Space

Keep the search space defined before final evaluation.

Example:


In [ ]:
hyperparameter_grid = [
    {
        "learning_rate":
            1e-3,

        "weight_decay":
            1e-4
    },
    {
        "learning_rate":
            3e-4,

        "weight_decay":
            1e-4
    },
    {
        "learning_rate":
            1e-3,

        "weight_decay":
            1e-3
    }
]

print(
    hyperparameter_grid
)


# 40. Do Not Tune Every Possible Detail

If you repeatedly try:

- Many crops
- Many augmentations
- Many normalization methods
- Many architectures
- Many thresholds

on the same validation patients, you gradually overfit the validation set.

Validation overfitting is real.


# 41. Ablation Studies

An ablation study asks:

> Which component actually contributes to performance?

Start with a full system.

Then remove or replace one component at a time.

Example:

$$
Full\ Model
$$

versus:

- No augmentation
- No weight decay
- No harmonization
- No class weighting
- Smaller network


# 42. One-Factor-at-a-Time Ablations

A good ablation changes one factor while keeping fixed:

- Patient folds
- Seeds
- Training epochs
- Architecture, unless architecture is the factor
- Metric definitions
- Checkpointing


In [ ]:
ablation_configs = {
    "baseline": {
        **baseline_config
    },

    "no_augmentation": {
        **baseline_config,
        "name":
            "no_augmentation",
        "augmentation":
            False
    },

    "no_weight_decay": {
        **baseline_config,
        "name":
            "no_weight_decay",
        "weight_decay":
            0.0
    },

    "per_image_normalization": {
        **baseline_config,
        "name":
            "per_image_normalization",
        "normalization":
            "per_image"
    },

    "no_normalization": {
        **baseline_config,
        "name":
            "no_normalization",
        "normalization":
            "none"
    }
}

print(
    ablation_configs.keys()
)


# 43. Fair Ablation Protocol

Every ablation should use the same:

$$
Fold\ IDs
$$

and:

$$
Random\ Seeds
$$

This creates a **paired comparison**.

Paired comparisons are usually more informative than comparing unrelated runs.


# 44. Quick Ablation Runner


In [ ]:
def run_ablation_suite(
    configs,
    internal_metadata,
    images,
    folds,
    seeds,
    epochs=2
):
    all_results = []
    all_predictions = []

    for name, config in (
        configs.items()
    ):
        print(
            "Running:",
            name
        )

        results, predictions = (
            run_cross_validation(
                internal_metadata,
                images,
                config,
                folds=folds,
                seeds=seeds,
                epochs=epochs
            )
        )

        all_results.append(
            results
        )

        all_predictions.append(
            predictions
        )

    return (
        pd.concat(
            all_results,
            ignore_index=True
        ),
        pd.concat(
            all_predictions,
            ignore_index=True
        )
    )


# 45. Lightweight Ablation Demonstration

To keep the notebook practical, the demo compares only:

- Baseline
- No augmentation

using two folds and one seed.

For research, run all predefined ablations across all folds/seeds.


In [ ]:
demo_ablation_configs = {
    "baseline":
        ablation_configs[
            "baseline"
        ],

    "no_augmentation":
        ablation_configs[
            "no_augmentation"
        ]
}

ablation_results, ablation_predictions = (
    run_ablation_suite(
        demo_ablation_configs,
        internal_metadata,
        research_images,
        folds=[
            0,
            1
        ],
        seeds=[
            11
        ],
        epochs=1
    )
)

print(
    ablation_results
)


# 46. Build an Ablation Result Table


In [ ]:
ablation_summary = (
    ablation_results
    .groupby(
        "experiment"
    )[
        [
            "val_accuracy",
            "val_macro_f1"
        ]
    ]
    .agg(
        [
            "mean",
            "std",
            "count"
        ]
    )
)

print(
    ablation_summary
)


# 47. Comparing Preprocessing Strategies Fairly

Suppose you compare:

- Global normalization
- Per-image normalization
- No normalization

You must keep fixed:

- Same patient folds
- Same seeds
- Same architecture
- Same optimizer
- Same training duration

Otherwise the result does not isolate preprocessing.


# 48. Comparing Harmonization Methods Fairly

A harmonization comparison should ideally use:

$$
\boxed{
Same\ downstream\ model
+
Same\ folds
+
Same\ seeds
+
Same\ metrics
}
$$

Only the harmonization component should change.

And every harmonization method must be fitted using fold-training data only.


# 49. Leakage-Safe Harmonization Inside CV

For fold $k$:

1. Fit harmonization on fold-training patients
2. Apply frozen harmonization to fold-validation patients
3. Train model
4. Evaluate
5. Repeat for every fold

Do not fit harmonization once using all development patients before cross-validation.


# 50. Repeated Experiments With Multiple Seeds

When comparing two methods, use the same seed set:

$$
\{11,22,33,\ldots\}
$$

for both.

This makes the comparison more paired and easier to interpret.


# 51. Build a Reproducible Experiment ID

A stable experiment ID can be generated from the configuration.


In [ ]:
def experiment_id(
    config
):
    text = json.dumps(
        config,
        sort_keys=True
    )

    digest = hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()

    return digest[
        :12
    ]


print(
    "Baseline ID:",
    experiment_id(
        baseline_config
    )
)


# 52. Experiment Tracking

At minimum, record:

- Experiment ID
- Config
- Fold
- Seed
- Validation metrics
- Best epoch
- Preprocessing statistics
- Checkpoint path
- Software version


# 53. Build a Long-Format Result Table

Long format is easy to:

- Group
- Filter
- Plot
- Aggregate
- Save to CSV


In [ ]:
tracked_results = (
    quick_cv_results
    .copy()
)

tracked_results[
    "experiment_id"
] = experiment_id(
    baseline_config
)

tracked_results[
    "torch_version"
] = torch.__version__

print(
    tracked_results.head()
)


# 54. Save Experiment Results


In [ ]:
RESULTS_DIR = Path(
    "research_grade_results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

tracked_results.to_csv(
    RESULTS_DIR
    / "cv_results.csv",
    index=False
)

quick_cv_predictions.to_csv(
    RESULTS_DIR
    / "cv_predictions.csv",
    index=False
)

(
    RESULTS_DIR
    / "baseline_config.json"
).write_text(
    json.dumps(
        baseline_config,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Research outputs saved."
)


# 55. Patient-Level Prediction Aggregation

Because each patient has multiple images, aggregate image probabilities before patient-level evaluation.

A simple strategy:

$$
\bar{p}_c
=
\frac{1}{M}
\sum_{m=1}^{M}
p_{m,c}
$$


In [ ]:
def aggregate_patient_predictions(
    prediction_df,
    num_classes=3
):
    probability_columns = [
        f"prob_class_{class_index}"
        for class_index in range(
            num_classes
        )
    ]

    rows = []

    for patient_id, group in (
        prediction_df.groupby(
            "patient_id"
        )
    ):
        true_labels = group[
            "true_label"
        ].unique()

        assert (
            len(
                true_labels
            )
            == 1
        )

        mean_probs = (
            group[
                probability_columns
            ]
            .mean()
        )

        predicted_label = int(
            mean_probs
            .to_numpy()
            .argmax()
        )

        row = {
            "patient_id":
                patient_id,

            "true_label":
                int(
                    true_labels[
                        0
                    ]
                ),

            "predicted_label":
                predicted_label
        }

        for column in (
            probability_columns
        ):
            row[
                column
            ] = float(
                mean_probs[
                    column
                ]
            )

        rows.append(
            row
        )

    return pd.DataFrame(
        rows
    )


# 56. One Prediction Per Patient for One Seed

Across CV folds, each internal patient is validated once per seed.

So for a single seed, we can combine all fold predictions into one patient-level validation set.


In [ ]:
seed_for_bootstrap = 11

seed_predictions = (
    quick_cv_predictions[
        quick_cv_predictions[
            "seed"
        ]
        == seed_for_bootstrap
    ]
    .reset_index(
        drop=True
    )
)

patient_predictions = (
    aggregate_patient_predictions(
        seed_predictions,
        num_classes=3
    )
)

print(
    patient_predictions.head()
)


# 57. Confidence Intervals

A point estimate such as:

$$
Accuracy=0.87
$$

does not communicate uncertainty.

A confidence interval provides a range of plausible values under the chosen resampling assumptions.

In patient-based studies, resampling should often happen at the patient level.


# 58. Patient-Level Bootstrap

Bootstrap procedure:

1. Start with $N$ patients
2. Sample $N$ patients **with replacement**
3. Compute the metric
4. Repeat many times
5. Use the bootstrap distribution to estimate uncertainty


# 59. Accuracy Metric for Patient Bootstrap


In [ ]:
def patient_accuracy(
    dataframe
):
    return float(
        (
            dataframe[
                "true_label"
            ].to_numpy()
            ==
            dataframe[
                "predicted_label"
            ].to_numpy()
        ).mean()
    )


print(
    "Patient accuracy:",
    patient_accuracy(
        patient_predictions
    )
)


# 60. Macro F1 Metric for Patient Bootstrap


In [ ]:
def patient_macro_f1(
    dataframe,
    num_classes=3
):
    return float(
        macro_f1_from_predictions(
            dataframe[
                "true_label"
            ].tolist(),
            dataframe[
                "predicted_label"
            ].tolist(),
            num_classes
        )
    )


print(
    "Patient macro F1:",
    patient_macro_f1(
        patient_predictions
    )
)


# 61. Bootstrap Confidence Interval Function


In [ ]:
def bootstrap_confidence_interval(
    patient_dataframe,
    metric_fn,
    n_bootstrap=1000,
    confidence_level=0.95,
    seed=42
):
    generator = (
        torch.Generator()
        .manual_seed(
            seed
        )
    )

    n_patients = len(
        patient_dataframe
    )

    values = []

    for _ in range(
        n_bootstrap
    ):
        sampled_indices = torch.randint(
            0,
            n_patients,
            (n_patients,),
            generator=generator
        ).tolist()

        sample = (
            patient_dataframe
            .iloc[
                sampled_indices
            ]
            .reset_index(
                drop=True
            )
        )

        values.append(
            metric_fn(
                sample
            )
        )

    values_tensor = torch.tensor(
        values,
        dtype=torch.float32
    )

    alpha = (
        1.0
        - confidence_level
    )

    lower = torch.quantile(
        values_tensor,
        alpha / 2
    ).item()

    upper = torch.quantile(
        values_tensor,
        1.0 - alpha / 2
    ).item()

    point_estimate = metric_fn(
        patient_dataframe
    )

    return {
        "estimate":
            float(
                point_estimate
            ),

        "lower":
            float(
                lower
            ),

        "upper":
            float(
                upper
            ),

        "confidence_level":
            confidence_level
    }


# 62. Bootstrap Accuracy Interval


In [ ]:
accuracy_ci = (
    bootstrap_confidence_interval(
        patient_predictions,
        patient_accuracy,
        n_bootstrap=500,
        confidence_level=0.95,
        seed=42
    )
)

print(
    accuracy_ci
)


# 63. Bootstrap Macro-F1 Interval


In [ ]:
macro_f1_ci = (
    bootstrap_confidence_interval(
        patient_predictions,
        lambda dataframe:
            patient_macro_f1(
                dataframe,
                num_classes=3
            ),
        n_bootstrap=500,
        confidence_level=0.95,
        seed=42
    )
)

print(
    macro_f1_ci
)


# 64. Why Bootstrap Patients Instead of Images?

If one patient contributes:

$$
10
$$

correlated images, image-level bootstrap treats those frames as if they were independent.

That can underestimate uncertainty.

Patient-level bootstrap respects the patient as the independent unit.


# 65. Confidence Intervals Do Not Fix Bias

A narrow confidence interval around a biased estimate is still biased.

Confidence intervals do not correct:

- Data leakage
- Poor external validity
- Label errors
- Selection bias
- Test-set tuning


# 66. External Validation

External validation asks:

> Does the selected pipeline generalize to patients from a different acquisition environment?

Our synthetic external cohort comes from:

- `External_Site`
- `Device_X`

It has never been used in cross-validation.


# 67. External Validation Must Remain Untouched

Do not use the external cohort to choose:

- Hyperparameters
- Harmonization method
- Architecture
- Seed
- Threshold
- Epoch count

If you tune on it repeatedly, it is no longer external validation.


# 68. Final Training After CV

After choosing the final configuration, one option is:

1. Fix the chosen hyperparameters
2. Choose training duration from CV
3. Train on all internal development patients
4. Fit preprocessing on all internal development data
5. Evaluate external cohort once


In [ ]:
def train_final_internal_model(
    internal_metadata,
    external_metadata,
    images,
    config,
    epochs=3,
    seed=42,
    num_classes=3
):
    set_seed(
        seed
    )

    train_mean, train_std = (
        compute_tensor_mean_std(
            internal_metadata,
            images
        )
    )

    train_dataset = (
        ResearchUltrasoundDataset(
            internal_metadata,
            images,
            training=config[
                "augmentation"
            ],
            normalization=config[
                "normalization"
            ],
            train_mean=train_mean,
            train_std=train_std
        )
    )

    external_dataset = (
        ResearchUltrasoundDataset(
            external_metadata,
            images,
            training=False,
            normalization=config[
                "normalization"
            ],
            train_mean=train_mean,
            train_std=train_std
        )
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=24,
        shuffle=True,
        num_workers=0
    )

    external_loader = DataLoader(
        external_dataset,
        batch_size=48,
        shuffle=False,
        num_workers=0
    )

    model = ResearchCNN(
        num_classes=num_classes,
        width=config[
            "width"
        ],
        dropout=config[
            "dropout"
        ]
    ).to(
        device
    )

    if config[
        "class_weighting"
    ]:
        weights = class_weights_from_metadata(
            internal_metadata,
            num_classes=num_classes
        ).to(
            device
        )

        criterion = nn.CrossEntropyLoss(
            weight=weights
        )

    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config[
            "learning_rate"
        ],
        weight_decay=config[
            "weight_decay"
        ]
    )

    for _ in range(
        epochs
    ):
        train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

    (
        external_loss,
        external_accuracy,
        external_macro_f1,
        external_predictions
    ) = evaluate_with_predictions(
        model,
        external_dataset,
        external_loader,
        criterion,
        device,
        num_classes=num_classes
    )

    return {
        "model":
            model,

        "train_mean":
            train_mean,

        "train_std":
            train_std,

        "external_loss":
            external_loss,

        "external_accuracy":
            external_accuracy,

        "external_macro_f1":
            external_macro_f1,

        "external_predictions":
            external_predictions
    }


# 69. External Validation Demonstration


In [ ]:
external_result = (
    train_final_internal_model(
        internal_metadata,
        external_metadata,
        research_images,
        baseline_config,
        epochs=2,
        seed=11
    )
)

print(
    "External accuracy:",
    external_result[
        "external_accuracy"
    ]
)

print(
    "External macro F1:",
    external_result[
        "external_macro_f1"
    ]
)


# 70. External Patient-Level Evaluation


In [ ]:
external_patient_predictions = (
    aggregate_patient_predictions(
        external_result[
            "external_predictions"
        ],
        num_classes=3
    )
)

print(
    "External patient accuracy:",
    patient_accuracy(
        external_patient_predictions
    )
)


# 71. External Confidence Interval


In [ ]:
external_accuracy_ci = (
    bootstrap_confidence_interval(
        external_patient_predictions,
        patient_accuracy,
        n_bootstrap=500,
        confidence_level=0.95,
        seed=123
    )
)

print(
    external_accuracy_ci
)


# 72. Statistical Model Comparison Intuition

Suppose Model A and Model B are evaluated on the **same patients**.

Their errors are paired.

A useful comparison asks:

$$
\boxed{
Metric_B
-
Metric_A
}
$$

and estimates uncertainty in that difference using the same patient resamples.


# 73. Why Paired Comparison Is Better

If patient P013 is difficult for both models, that shared difficulty should be preserved.

Paired analysis keeps the same sampled patients for both models.

This reduces irrelevant variation.


# 74. Paired Patient Bootstrap for Metric Difference

The function below assumes both model tables contain exactly one row per patient and the same patient IDs.


In [ ]:
def paired_bootstrap_difference(
    model_a,
    model_b,
    metric_fn,
    n_bootstrap=1000,
    seed=42
):
    merged_ids = sorted(
        set(
            model_a[
                "patient_id"
            ]
        )
        &
        set(
            model_b[
                "patient_id"
            ]
        )
    )

    a = (
        model_a
        .set_index(
            "patient_id"
        )
        .loc[
            merged_ids
        ]
        .reset_index()
    )

    b = (
        model_b
        .set_index(
            "patient_id"
        )
        .loc[
            merged_ids
        ]
        .reset_index()
    )

    assert (
        a[
            "patient_id"
        ].tolist()
        ==
        b[
            "patient_id"
        ].tolist()
    )

    n = len(
        a
    )

    generator = (
        torch.Generator()
        .manual_seed(
            seed
        )
    )

    differences = []

    for _ in range(
        n_bootstrap
    ):
        indices = torch.randint(
            0,
            n,
            (n,),
            generator=generator
        ).tolist()

        a_sample = a.iloc[
            indices
        ]

        b_sample = b.iloc[
            indices
        ]

        differences.append(
            metric_fn(
                b_sample
            )
            -
            metric_fn(
                a_sample
            )
        )

    difference_tensor = torch.tensor(
        differences,
        dtype=torch.float32
    )

    return {
        "point_difference":
            float(
                metric_fn(
                    b
                )
                -
                metric_fn(
                    a
                )
            ),

        "lower":
            float(
                torch.quantile(
                    difference_tensor,
                    0.025
                ).item()
            ),

        "upper":
            float(
                torch.quantile(
                    difference_tensor,
                    0.975
                ).item()
            )
    }


# 75. Interpreting a Difference Interval

Suppose the 95% bootstrap interval for:

$$
Accuracy_B-Accuracy_A
$$

is:

$$
[-0.01,\ 0.08]
$$

The observed difference may favor B, but the interval includes zero.

That indicates substantial uncertainty about whether B is truly better in the evaluated population.


# 76. Statistical Significance Is Not Clinical Significance

A tiny performance improvement can be statistically detectable but clinically irrelevant.

Always ask:

- How large is the effect?
- Is it clinically meaningful?
- Is the confidence interval narrow enough?
- Does it generalize externally?


# 77. Cross-Validation Folds Are Not Fully Independent Experiments

Training sets overlap heavily across folds.

Therefore naïvely treating fold scores as independent samples in a standard t-test can be misleading.

Use statistical methods appropriate for paired/repeated model evaluation.

For serious claims, consult the statistical methodology relevant to your metric and study design.


# 78. AUROC Comparisons

For AUROC, common comparison approaches include:

- Paired bootstrap
- DeLong-style methods for paired ROC curves

The right method depends on:

- Same vs different patients
- Binary vs multi-class task
- Resampling design

This notebook focuses on the general paired-resampling intuition.


# 79. Experiment Tracking Directory

A research project might organize runs as:

```text
experiments/
│
├── exp_001/
│   ├── config.json
│   ├── fold_results.csv
│   ├── predictions.csv
│   └── checkpoints/
│
├── exp_002/
└── summary/
```


# 80. Reproducible Result Tables

A useful result table can contain one row per:

$$
Experiment
\times
Fold
\times
Seed
$$

Columns may include:

- Experiment ID
- Model
- Preprocessing
- Harmonization
- Fold
- Seed
- Validation loss
- Accuracy
- Macro F1
- Best epoch


In [ ]:
result_table = tracked_results[
    [
        "experiment_id",
        "experiment",
        "fold",
        "seed",
        "best_epoch",
        "val_loss",
        "val_accuracy",
        "val_macro_f1"
    ]
].copy()

print(
    result_table
)


# 81. Aggregate Result Table


In [ ]:
aggregate_result_table = (
    result_table
    .groupby(
        [
            "experiment_id",
            "experiment"
        ]
    )
    .agg(
        mean_accuracy=(
            "val_accuracy",
            "mean"
        ),

        std_accuracy=(
            "val_accuracy",
            "std"
        ),

        mean_macro_f1=(
            "val_macro_f1",
            "mean"
        ),

        std_macro_f1=(
            "val_macro_f1",
            "std"
        ),

        runs=(
            "val_accuracy",
            "count"
        )
    )
    .reset_index()
)

print(
    aggregate_result_table
)


# 82. Avoid Reporting the Maximum Fold Score

Wrong:

> Our model achieved 98% accuracy.

when 98% was only the best fold.

Better:

> Mean patient-level performance across the predefined folds/seeds was X, with variability Y.

The reporting unit must match the experimental protocol.


# 83. Repeated CV Reporting

Possible report:

> 5-fold patient-grouped CV repeated over 3 random seeds.

Then report:

- Mean fold/seed performance
- Standard deviation
- Patient-level pooled predictions when appropriate
- Confidence intervals


# 84. Pooled Out-of-Fold Predictions

Cross-validation produces an important artifact:

> **Out-of-fold predictions**

Every patient receives predictions from a model that was not trained on that patient.

These predictions can support:

- Error analysis
- Calibration analysis
- Threshold selection
- Patient-level bootstrap

provided the procedure remains leakage-safe.


# 85. Why Out-of-Fold Predictions Are Valuable

Out-of-fold predictions approximate development-set generalization more honestly than training predictions.

They can be saved as:

```text
patient_id,true_label,probability,...
```

and reused for later analyses.


# 86. Hyperparameter Selection and Out-of-Fold Predictions

Be careful:

If you select a hyperparameter using all outer-fold out-of-fold scores and then report those same scores as final unbiased performance, some selection optimism remains.

For strong performance estimation:

- Nested CV
- Or a completely separate test/external cohort

is preferable.


# 87. External Validation Is Stronger Than Internal CV for Domain Shift

Cross-validation answers:

> Does the model generalize to held-out patients from the same development distribution?

External validation asks:

> Does the model generalize to a new acquisition environment?

These are different questions.


# 88. Site-Held-Out Validation

If you have multiple sites, another powerful design is:

> Leave one site out entirely.

Train on:

- Site A
- Site B

Test on:

- Site C

This directly studies site-domain shift.


# 89. Device-Held-Out Validation

Similarly:

> Hold one scanner/device family out of training.

This evaluates robustness to acquisition hardware changes.

It can be more difficult—but more informative—than random patient-grouped CV.


# 90. Repeated Seeds Are Not a Substitute for More Patients

Running:

$$
100
$$

seeds on a tiny dataset does not create new biological diversity.

Seed repetitions quantify training stochasticity.

They do not replace independent patients.


# 91. More Folds Are Not Always Better

Increasing:

$$
K
$$

means:

- Larger training fraction per fold
- Smaller validation fold
- More training runs

Very small validation folds can produce noisy metrics.

Choose the number of folds based on:

- Dataset size
- Class counts
- Patient groups
- Compute budget


# 92. Rare Classes and Fold Construction

Every fold should ideally contain enough examples of every clinically important class.

If one class has only:

$$
4
$$

patients, 5-fold stratification cannot place that class into all folds.

This is a dataset limitation, not a software problem.


# 93. Hyperparameter Search Should Respect Patient Groups

Even inside the inner loop:

$$
Patient
$$

must remain the grouping unit.

Do not use image-level random validation inside a patient-grouped outer loop.


# 94. Ablations Should Answer a Scientific Question

Bad ablation:

> Try random changes and report whichever helps.

Good ablation:

> Does device-aware normalization improve external-device generalization compared with global normalization?

An ablation should test a hypothesis.


# 95. Harmonization Ablation Questions

Examples:

- Does harmonization improve external-site AUROC?
- Does it reduce device-performance variance?
- Does it improve calibration?
- Does it hurt one disease class?
- Does it reduce scanner-focused saliency?

These are stronger questions than:

> Which image looks cleaner?


# 96. External Validation Sample Size

An external cohort with:

$$
10
$$

patients can produce very wide uncertainty intervals.

Always report:

- Number of patients
- Number of events/classes
- Confidence intervals

Performance without sample size is incomplete.


# 97. Patient-Level Bootstrap With Multiple Images

If your final evaluation is patient-level, aggregate images first.

Then bootstrap patient rows.

Do not bootstrap individual image frames after aggregation rules were selected.


# 98. Save the Full Protocol Before Final Testing

A strong protocol file records:

- Primary metric
- Fold construction
- Seeds
- Hyperparameter search space
- Ablations
- Model-selection rule
- External-test definition
- Bootstrap method


In [ ]:
research_protocol = {
    "group_unit":
        "patient",

    "internal_cv":
        {
            "folds":
                5,

            "stratified_by":
                "label"
        },

    "training_seeds":
        [
            11,
            22,
            33
        ],

    "primary_metric":
        "patient_macro_f1",

    "model_selection":
        "inner_validation_only",

    "external_validation":
        "External_Site",

    "confidence_interval":
        "patient_level_bootstrap"
}

(
    RESULTS_DIR
    / "research_protocol.json"
).write_text(
    json.dumps(
        research_protocol,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Protocol saved."
)


# 99. Research-Quality Reporting Checklist

A paper/report should state:

1. Number of patients
2. Number of images
3. Class distribution
4. Number of sites/devices
5. Split unit
6. Cross-validation scheme
7. Number of seeds/repetitions
8. Preprocessing
9. Harmonization
10. Architecture
11. Hyperparameter tuning procedure
12. Primary metric
13. Confidence intervals
14. External validation
15. Patient-level aggregation rule


# 100. Example Methods-Section Language

A clear report might say:

> We used 5-fold stratified patient-grouped cross-validation. All images from a patient were assigned to the same fold. Preprocessing statistics were estimated independently within each training fold. Hyperparameters were selected using inner patient-grouped validation. Final external evaluation was performed on a held-out site that was not used during development.

The exact wording should match what you actually did.


# 101. Common Mistake — Reusing One Validation Fold for Everything

If the same validation set is used to choose:

- Architecture
- Augmentation
- Harmonization
- Learning rate
- Threshold
- Epoch count

for many experiments, you eventually overfit that validation set.


# 102. Common Mistake — Computing Normalization Before CV

Wrong:

```text
all internal patients
→ compute mean/std
→ create folds
```

Better:

```text
create folds
→ for each fold:
    fit mean/std on fold-training patients
```


# 103. Common Mistake — Harmonization Before Fold Assignment

A learned harmonization model trained on all internal data has already seen validation-fold information.

Fit harmonization separately inside each training fold.


# 104. Common Mistake — Mixing Different Seeds Between Methods

If Method A uses lucky seeds and Method B uses different unlucky seeds, the comparison contains unnecessary noise.

Use the same predefined seed set.


# 105. Common Mistake — Comparing Different Patient Folds

Keep patient folds identical across competing methods.

Otherwise differences may come from patient difficulty rather than the method.


# 106. Common Mistake — Reporting Mean Without Variability

A mean without:

- Standard deviation
- Confidence interval
- Number of patients/runs

hides uncertainty.


# 107. Common Mistake — Treating Fold Standard Deviation as a Confidence Interval

Fold-to-fold standard deviation and a patient-level confidence interval answer different questions.

Do not label one as the other.


# 108. Common Mistake — Bootstrapping Images Instead of Patients

If images are clustered within patients, image-level bootstrap violates the independence assumption.

Resample the patient unit.


# 109. Common Mistake — Selecting the Best Seed

Do not report only the best random seed.

Seeds are not hyperparameters to cherry-pick.

Report across the predefined seed set.


# 110. Common Mistake — Tuning on External Validation

Once external data influences model decisions, it becomes part of development.

You need another untouched cohort for truly independent validation.


# 111. Common Mistake — Treating Statistical Significance as Model Quality

A p-value does not tell you:

- Clinical value
- Calibration
- Robustness
- External generalization
- Bias

Statistical comparison is one component of evidence.


# 112. Common Mistake — Running Many Ablations Without Hypotheses

Ablation studies should be planned to answer specific questions.

Too many post-hoc comparisons increase the risk of misleading conclusions.


# 113. Practice Exercises

## Exercise 1

Create 5 patient-grouped stratified folds for a patient table.

## Exercise 2

Verify that every patient belongs to exactly one fold.

## Exercise 3

For one validation fold, compute normalization statistics using training patients only.

## Exercise 4

Run the same fold using two different random seeds.

## Exercise 5

Create an ablation that removes augmentation while keeping everything else fixed.

## Exercise 6

Create a nested-validation patient table for one outer fold.

## Exercise 7

Aggregate image probabilities into patient-level predictions.

## Exercise 8

Compute a 95% patient-level bootstrap confidence interval for accuracy.

## Exercise 9

Create a paired bootstrap function for the difference between two models.

## Exercise 10

Build a CSV result table with one row per experiment × fold × seed.


# 114. Conceptual Challenges

## Challenge 1

Why can one train/validation split be unstable?

## Challenge 2

Why must all images from one patient remain in one fold?

## Challenge 3

What is the difference between grouped CV and stratified grouped CV?

## Challenge 4

Why must normalization be fitted separately inside every fold?

## Challenge 5

What does repeating several seeds measure?

## Challenge 6

Why are multiple seeds not a substitute for more patients?

## Challenge 7

What is the purpose of nested validation?

## Challenge 8

Why can repeated tuning overfit a validation set?

## Challenge 9

What makes an ablation comparison fair?

## Challenge 10

Why must harmonization be fitted inside the training portion of each fold?

## Challenge 11

Why should confidence intervals often resample patients rather than images?

## Challenge 12

Why is external-site validation different from internal cross-validation?

## Challenge 13

Why should paired model comparisons use the same patient samples?

## Challenge 14

Why should the best random seed not be reported as the final result?

## Challenge 15

What information belongs in a research-grade result table?


# 115. Exercise Solutions


In [ ]:
# Exercise 1
exercise_patient_table = (
    patient_table[
        [
            "patient_id",
            "label",
            "site",
            "device"
        ]
    ]
    .copy()
)

exercise_fold_map = (
    make_stratified_group_folds(
        exercise_patient_table,
        n_splits=5,
        seed=7
    )
)

exercise_patient_table[
    "fold"
] = exercise_patient_table[
    "patient_id"
].map(
    exercise_fold_map
)

print(
    pd.crosstab(
        exercise_patient_table[
            "fold"
        ],
        exercise_patient_table[
            "label"
        ]
    )
)


In [ ]:
# Exercise 2
exercise_fold_counts = (
    exercise_patient_table
    .groupby(
        "patient_id"
    )[
        "fold"
    ]
    .nunique()
)

assert (
    exercise_fold_counts
    == 1
).all()

print(
    "Every patient belongs to one fold."
)


In [ ]:
# Exercise 3
exercise_val_fold = 0

exercise_train_metadata = (
    internal_metadata[
        internal_metadata[
            "fold"
        ]
        != exercise_val_fold
    ]
)

exercise_mean, exercise_std = (
    compute_tensor_mean_std(
        exercise_train_metadata,
        research_images
    )
)

print(
    exercise_mean,
    exercise_std
)


In [ ]:
# Exercise 4
seed_1_result, _ = (
    run_single_fold(
        internal_metadata,
        research_images,
        val_fold=0,
        seed=101,
        config=baseline_config,
        epochs=1
    )
)

seed_2_result, _ = (
    run_single_fold(
        internal_metadata,
        research_images,
        val_fold=0,
        seed=202,
        config=baseline_config,
        epochs=1
    )
)

print(
    seed_1_result[
        "val_accuracy"
    ],
    seed_2_result[
        "val_accuracy"
    ]
)


In [ ]:
# Exercise 5
exercise_no_aug = {
    **baseline_config,
    "name":
        "exercise_no_augmentation",
    "augmentation":
        False
}

print(
    exercise_no_aug
)


In [ ]:
# Exercise 6
exercise_outer_train, exercise_outer_val = (
    make_nested_patient_tables(
        patient_table,
        outer_fold=1,
        n_inner_splits=3,
        seed=99
    )
)

print(
    "Outer train patients:",
    len(
        exercise_outer_train
    )
)

print(
    "Outer validation patients:",
    len(
        exercise_outer_val
    )
)


In [ ]:
# Exercise 7
exercise_patient_predictions = (
    aggregate_patient_predictions(
        seed_predictions,
        num_classes=3
    )
)

print(
    exercise_patient_predictions.head()
)


In [ ]:
# Exercise 8
exercise_ci = (
    bootstrap_confidence_interval(
        exercise_patient_predictions,
        patient_accuracy,
        n_bootstrap=300,
        confidence_level=0.95,
        seed=7
    )
)

print(
    exercise_ci
)


In [ ]:
# Exercise 9
# Function already implemented:
print(
    paired_bootstrap_difference
)


In [ ]:
# Exercise 10
exercise_result_table = (
    quick_cv_results[
        [
            "experiment",
            "fold",
            "seed",
            "val_loss",
            "val_accuracy",
            "val_macro_f1"
        ]
    ]
    .copy()
)

exercise_result_table.to_csv(
    "exercise_result_table.csv",
    index=False
)

print(
    exercise_result_table.head()
)


# 116. Conceptual Challenge Solutions

## Challenge 1

One split may accidentally contain unusually easy or difficult patients, rare classes, or acquisition patterns. Small datasets are especially sensitive to this.

## Challenge 2

Images from the same patient are correlated. Splitting them across folds can leak patient-specific information and inflate validation performance.

## Challenge 3

Grouped CV preserves patient independence. Stratified grouped CV additionally tries to keep class distributions similar across folds.

## Challenge 4

Otherwise the validation fold influences preprocessing parameters such as mean, standard deviation, or harmonization statistics.

## Challenge 5

Repeated seeds measure stochastic variation caused by initialization, mini-batch order, augmentation, and optimization.

## Challenge 6

Seeds do not create new biological or acquisition diversity. Only additional independent patients provide new samples from the target population.

## Challenge 7

Nested validation separates hyperparameter selection from outer-fold performance estimation.

## Challenge 8

Repeatedly choosing methods that score well on the same validation patients gradually adapts the development process to those patients.

## Challenge 9

A fair ablation uses the same folds, seeds, training schedule, metrics, and downstream pipeline while changing only the component being studied.

## Challenge 10

A learned harmonization transform trained on validation-fold data leaks information into the training pipeline.

## Challenge 11

The patient is often the independent unit. Images from one patient are correlated, so resampling images can underestimate uncertainty.

## Challenge 12

Internal CV tests held-out patients from the development distribution. External validation tests a different site/device/acquisition environment.

## Challenge 13

Paired analysis preserves the same patient difficulty for both models and focuses the comparison on the method difference.

## Challenge 14

Choosing the best seed is cherry-picking stochastic noise. Report performance across the predefined seed set.

## Challenge 15

A result table should identify the experiment, configuration, fold, seed, metrics, best epoch, and any preprocessing/harmonization details needed to reproduce the run.


# 117. Key Takeaways

In this notebook, we studied:

- Why one validation split may be unstable
- Patient-grouped cross-validation
- Stratified patient folds
- Fold-specific preprocessing
- Repeated experiments with multiple seeds
- Hyperparameter tuning without leakage
- Nested-validation intuition
- Ablation studies
- Fair preprocessing comparisons
- Fair harmonization comparisons
- Experiment IDs
- Experiment tracking
- Long-format result tables
- Out-of-fold predictions
- Patient-level aggregation
- Confidence intervals
- Patient-level bootstrap
- Paired bootstrap comparison
- External validation
- Site-held-out evaluation
- Device-held-out evaluation
- Statistical comparison intuition
- Research reporting principles

The most important cross-validation rule is:

$$
\boxed{
Patient\ Grouping
}
$$

The most important tuning rule is:

$$
\boxed{
Never\ Tune\ on\ the\ Evaluation\ Fold
}
$$

The most important uncertainty rule is:

$$
\boxed{
Bootstrap\ the\ Independent\ Unit
}
$$

And the most important research principle is:

$$
\boxed{
Same\ Patients
+
Same\ Folds
+
Same\ Seeds
+
One\ Controlled\ Change
}
$$

for fair model comparisons.


# 118. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. Why can a single validation split be unstable?
2. What is patient-grouped cross-validation?
3. Why is image-level K-fold risky for multi-image patients?
4. What does stratification try to preserve?
5. Why should every fold be inspected manually?
6. Why must preprocessing be fitted inside every fold?
7. Why should harmonization also be fold-specific?
8. What does repeating random seeds measure?
9. How many training runs are required for 5 folds × 3 seeds?
10. Why does repeated validation tuning cause optimism?
11. What is nested validation?
12. What is the outer loop used for?
13. What is the inner loop used for?
14. What is an ablation study?
15. What makes an ablation fair?
16. Why should competing methods use identical patient folds?
17. Why should competing methods use identical seed sets?
18. What is an out-of-fold prediction?
19. Why are out-of-fold predictions useful?
20. Why aggregate images to patient level before patient-level bootstrap?
21. What does a confidence interval communicate?
22. Why does a confidence interval not fix dataset bias?
23. What is patient-level bootstrap?
24. What is a paired bootstrap comparison?
25. Why is external validation different from internal CV?
26. What is site-held-out validation?
27. What is device-held-out validation?
28. Why should the best seed not be reported?
29. Why are fold scores not completely independent observations?
30. What information should be reported for a research-quality experiment?


# Next Notebook

# 26 — Robustness, Domain Shift, and Ultrasound Harmonization in Depth

In the next notebook, we will study:

- What is domain shift?
- Scanner and site shift
- Covariate shift
- Label shift intuition
- Measuring distribution differences
- Site-held-out evaluation
- Device-held-out evaluation
- Robustness stress tests
- Intensity perturbation tests
- Resolution and noise sensitivity
- Harmonization goals
- Global vs per-image normalization
- Histogram-based harmonization
- Feature-space harmonization intuition
- Domain-invariant representation learning
- Detecting shortcut learning across sites
- Evaluating whether harmonization truly improves generalization
